In [11]:
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple, Union
import pandas as pd
import sys
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root)) 

In [12]:
def load_gold(gold_paths: List[Path]) -> pd.DataFrame:
    dfs = []

    for path in gold_paths:
        df = pd.read_json(path)[['id_', 'Question', 'table_names', 'sql_columns_by_table']].copy()

        if "two_table" in path.name:
            prefix = "MMQA-two_table"
        elif "three_table" in path.name:
            prefix = "MMQA-three_table"
        else:
            prefix = "MMQA"

        df['id_'] = df['id_'].apply(lambda x: f"{prefix}-{x}")

        dfs.append(df)

    total_df = pd.concat(dfs, ignore_index=True)
    print(f"Total loaded {len(total_df)} instances. Two-table: {len(dfs[0])}, Three-table: {len(dfs[1])}.")
    return total_df

In [13]:
def generate_predictions_paths(
    llm_lists: List[str],
    shot_methods: List[str],
    schema_linking_methods: List[str],
    dataset_name: str = 'MMQA',
    logs_root: Optional[Path] = None,
    keep_all_runs: bool = False,
)-> Dict:
    """按模型、shot方法、schemalinking方法生成预测日志路径。

    默认每个组合只返回最新一次运行的文件；若 keep_all_runs=True, 返回该组合的全部文件列表。
    """
    if logs_root is None:
        logs_root = project_root / 'Logs'

    predictions_paths: Dict[str, Dict[str, Dict[str, Union[Optional[Path], List[Path]]]]] = {}
    total_num = 0 

    for llm in llm_lists:
        model_dir = Path(logs_root / llm)
        predictions_paths[llm] = {}

        for st_method in shot_methods:
            predictions_paths[llm][st_method] = {}

            for sl_method in schema_linking_methods:
                pattern = f"{st_method}_{sl_method}_{dataset_name}_*.json"
                candidates = sorted(model_dir.glob(pattern)) if model_dir.exists() else []

                if keep_all_runs:
                    predictions_paths[llm][st_method][sl_method] = candidates
                else:
                    predictions_paths[llm][st_method][sl_method] = candidates[-1] if candidates else None
                total_num = total_num + len(candidates)
    print(f"Total loaded {total_num} logs.")
    return predictions_paths

In [14]:
def parse_response(text):
    try:
        response_dict = json.loads(text.strip())
        columns = response_dict.get("relevant_columns", {})
        tables = list(columns.keys())
        return tables, columns
    except (json.JSONDecodeError, TypeError, AttributeError):
        return [], {}

def parse_tables(text):
    try:
        data = json.loads(text.strip())
        return data.get("relevant_tables", [])
    except (json.JSONDecodeError, TypeError, AttributeError):
        return []
    
def parse_columns(text):
    try:
        data = json.loads(text.strip())
        return data.get("relevant_columns", {})
    except (json.JSONDecodeError, TypeError, AttributeError):
        return {}

In [42]:
# === Metrics: Exact/Partial + Group by table count ===
def _infer_group_from_id(sample_id: str) -> str:
    sid = str(sample_id)
    if 'two_table' in sid:
        return 'two_table'
    if 'three_table' in sid:
        return 'three_table'
    return 'unknown'

def _to_table_set(tables: Any) -> Set[str]:
    if isinstance(tables, list):
        return set(str(t) for t in tables)
    return set()

def _to_column_pair_set(columns: Any) -> Set[Tuple[str, str]]:
    pairs: Set[Tuple[str, str]] = set()
    if not isinstance(columns, dict):
        return pairs
    for table, cols in columns.items():
        if isinstance(cols, list):
            for col in cols:
                pairs.add((str(table), str(col)))
    return pairs



def _extract_prediction_df(pred_path: Path, sl_method: str) -> pd.DataFrame:
    df = pd.read_json(pred_path).copy()

    if sl_method == 'baseline_schema_linking':
        parsed = df['response'].apply(parse_response)
        df['tables_pred'] = parsed.apply(lambda x: x[0])
        df['columns_pred'] = parsed.apply(lambda x: x[1])
    elif sl_method == 'table2column':
        df['tables_pred'] = df['tables_from_llm'].apply(parse_tables)
        df['columns_pred'] = df['columns_from_llm'].apply(parse_columns)
    else:
        raise ValueError(f'Unsupported schema linking method: {sl_method}')

    return df[['id', 'tables_pred', 'columns_pred']]

def _micro_set_prf(gold_sets: Sequence[Set[Any]], pred_sets: Sequence[Set[Any]]) -> Dict[str, float]:
    tp = 0
    pred_total = 0
    gold_total = 0

    for gold_set, pred_set in zip(gold_sets, pred_sets):
        tp += len(gold_set & pred_set)
        pred_total += len(pred_set)
        gold_total += len(gold_set)

    precision = tp / pred_total if pred_total else 0.0
    recall = tp / gold_total if gold_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {'precision': round(precision * 100, 2), 'recall': round(recall * 100, 2), 'f1': round(f1 * 100, 2)}
    
def evaluate_one_run(gold_df: pd.DataFrame, pred_path: Path, sl_method: str) -> pd.DataFrame:
    pred_df = _extract_prediction_df(pred_path, sl_method)

    eval_df = gold_df[['id_', 'table_names', 'sql_columns_by_table']].copy()
    eval_df = eval_df.rename(columns={
        'id_': 'id',
        'table_names': 'tables_gold',
        'sql_columns_by_table': 'columns_gold'
    })
    eval_df['table_group'] = eval_df['id'].apply(_infer_group_from_id)

    merged = eval_df.merge(pred_df, on='id', how='left')

    merged['tables_gold_set'] = merged['tables_gold'].apply(_to_table_set)
    merged['tables_pred_set'] = merged['tables_pred'].apply(_to_table_set)
    merged['cols_gold_set'] = merged['columns_gold'].apply(_to_column_pair_set)
    merged['cols_pred_set'] = merged['columns_pred'].apply(_to_column_pair_set)

    out_rows = []
    groups = ['overall', 'two_table', 'three_table']

    for group in groups:
        subset = merged if group == 'overall' else merged[merged['table_group'] == group]
        if subset.empty:
            continue

        table_m = _micro_set_prf(subset['tables_gold_set'].tolist(), subset['tables_pred_set'].tolist())
        col_m = _micro_set_prf(subset['cols_gold_set'].tolist(), subset['cols_pred_set'].tolist())

        out_rows.append({
            'table_group': group,
            'samples': int(len(subset)),
            'table_precision': table_m['precision'],
            'table_recall': table_m['recall'],
            'table_f1': table_m['f1'],
            'column_precision': col_m['precision'],
            'column_recall': col_m['recall'],
            'column_f1': col_m['f1'],
        })

    return pd.DataFrame(out_rows)


In [40]:
def find_best_methods(df: pd.DataFrame) -> pd.DataFrame:
    """
    Find the best schema linking method (by table_f1) for each LLM
    under each table_group (overall, two_table, three_table).
    """
    
    # 找到每个 (llm, table_group) table_f1 最大的行索引
    idx = df.groupby(['llm', 'table_group'])['table_f1'].idxmax()
    
    # 取出这些行
    best_df = df.loc[idx].reset_index(drop=True)
    
    # 排序方便阅读
    best_df = best_df.sort_values(['llm', 'table_group'])
    
    return best_df

In [16]:
gold_paths = [
    project_root / 'Data' / 'MMQA' / 'Synthesized_two_table_with_spider_db_id.json',
    project_root / 'Data' / 'MMQA' / 'Synthesized_three_table_with_spider_db_id.json',
]

In [17]:
gold_df = load_gold(gold_paths)


Total loaded 3313 instances. Two-table: 2592, Three-table: 721.


In [20]:
# llm_lists = ['ministral-3:3b','mistral:7b','gemma3:1b','gemma3:4b','llama3.2:1b','llama3.2:3b','gpt-3.5-turbo', 'gpt-4o-mini']
llm_lists = ['ministral-3:3b','mistral:7b','gemma3:1b','gemma3:4b','llama3.2:1b','llama3.2:3b','gpt-3.5-turbo', 'gpt-4o-mini']
shot_methods = ['zero_shot','few_shot']
schema_linking_methods = ['baseline_schema_linking','table2column']
prediction_paths = generate_predictions_paths(llm_lists,shot_methods,schema_linking_methods)

Total loaded 32 logs.


In [43]:
# 运行所有预测文件并汇总结果
all_results = []
for llm, st_map in prediction_paths.items():
    for shot_method, sl_map in st_map.items():
        for sl_method, pred_path in sl_map.items():
            if pred_path is None:
                continue
            run_df = evaluate_one_run(gold_df, pred_path, sl_method)
            run_df.insert(0, 'schema_linking_method', sl_method)
            run_df.insert(0, 'shot_method', shot_method)
            run_df.insert(0, 'llm', llm)
            all_results.append(run_df)

results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
if not results_df.empty:
    score_cols = [
        'table_precision', 'table_recall', 'table_f1',
        'column_precision', 'column_recall', 'column_f1'
    ]
    results_df[score_cols] = results_df[score_cols].round(4)

    # display_cols = [
    #     'llm', 'shot_method', 'schema_linking_method', 'match_type', 'table_group', 'samples',
    #     'table_precision', 'table_recall', 'table_f1',
    #     'column_precision', 'column_recall', 'column_f1'
    # ]
    display_cols = [
        'llm', 'shot_method', 'schema_linking_method', 
        'table_group',
        'table_precision', 'table_recall', 'table_f1'
    ]
    print(results_df[display_cols].to_markdown(index=False))
else:
    print('No prediction files found in prediction_paths.')


| llm            | shot_method   | schema_linking_method   | table_group   |   table_precision |   table_recall |   table_f1 |
|:---------------|:--------------|:------------------------|:--------------|------------------:|---------------:|-----------:|
| ministral-3:3b | zero_shot     | baseline_schema_linking | overall       |             88.94 |          88.82 |      88.88 |
| ministral-3:3b | zero_shot     | baseline_schema_linking | two_table     |             87.51 |          92.98 |      90.16 |
| ministral-3:3b | zero_shot     | baseline_schema_linking | three_table   |             93.07 |          79.19 |      85.57 |
| ministral-3:3b | zero_shot     | table2column            | overall       |             89.9  |          86.06 |      87.93 |
| ministral-3:3b | zero_shot     | table2column            | two_table     |             88.56 |          90.64 |      89.59 |
| ministral-3:3b | zero_shot     | table2column            | three_table   |             93.83 |          75.44

In [44]:
results_df.to_csv('results.csv',index=False)

In [37]:
display_cols = [
        'llm', 'shot_method', 'schema_linking_method', 
        'table_group',
        'table_precision', 'table_recall', 'table_f1'
    ]
best_results = find_best_methods(results_df)[display_cols]

print(best_results.to_markdown(index=False))

| llm            | shot_method   | schema_linking_method   | table_group   |   table_precision |   table_recall |   table_f1 |
|:---------------|:--------------|:------------------------|:--------------|------------------:|---------------:|-----------:|
| gemma3:1b      | zero_shot     | table2column            | overall       |            0.6503 |         0.6012 |     0.6248 |
| gemma3:1b      | zero_shot     | table2column            | three_table   |            0.741  |         0.5123 |     0.6058 |
| gemma3:1b      | zero_shot     | table2column            | two_table     |            0.6239 |         0.6397 |     0.6317 |
| gemma3:4b      | few_shot      | table2column            | overall       |            0.8241 |         0.9008 |     0.8608 |
| gemma3:4b      | zero_shot     | table2column            | three_table   |            0.8785 |         0.9174 |     0.8975 |
| gemma3:4b      | few_shot      | baseline_schema_linking | two_table     |            0.8544 |         0.8933